# Firing-Structure Router (M4): firing-Jaccard predicts when *grouping* beats *attribution*

This demo accompanies the experiment **"Firing-Structure Router (M4): firing-Jaccard predicts when grouping beats attribution."**

## The idea in one paragraph

Single SAE latents are unreliable units of analysis (feature *absorption*, *splitting*, non-atomicity). A repair is to **group** related latents into a cluster-level unit (a label-free **CCRG** unit: a content-responsive *parent* anchor + firing-disjoint, hole-covering *absorber* latents). But grouping does not always help. This experiment promotes one cheap, **a-priori, label-free** measurement to a headline **router**: the **positive-only firing-Jaccard** between per-sub-context *detector* latents and the general *parent* latent.

Two regimes emerge:

- **Absorption regime** (firing-Jaccard ≪ τ): the concept fragments into a broad parent + narrow, *firing-disjoint* specialists. Re-grouping them (CCRG) **helps** → the label-free unit beats the best single raw latent.
- **Co-firing / splitting regime** (firing-Jaccard ≫ τ): sub-contexts ride on broad latents that *co-fire* with the parent; no disjoint absorber exists, so a **single specialist latent already wins** and marginal attribution is enough.

The router predicts the regime **before** any downstream outcome is revealed, is validated **prospectively** on 3 held-out concepts, and its honest failure mode is reported (firing-Jaccard *alone* is insufficient — it must co-occur with parent **recall holes**).

## What this notebook runs (and what it doesn't)

The full `method.py` is a single-GPU experiment: it loads `unsloth/gemma-2-2b` + a frozen **Gemma-Scope** SAE (`layer_12/width_16k/average_l0_82`), runs forward passes for **15 concepts**, and for each computes (1) the parent latent, (2) per-sub-context detectors + recall holes, (3) the firing-Jaccard, and (4) a matched-pool downstream **outcome** (label-free CCRG unit vs three supervised baselines). Those GPU steps are **cached** — their output is the per-concept firing-structure table.

This demo loads that precomputed table and **re-runs the headline contribution verbatim** on CPU in milliseconds: derive the router threshold τ\\*, the regime-separation margin, leave-one-concept-out (LOO) cross-validation, the prospective predict-then-measure test, and the alternative (recall-hole-only / 2-signal) routers. We then check the re-derived numbers match the original GPU run.

**Concepts:** 12 ESTABLISHED (spelling L/O/T/I/D; numeric; taxonomic; toxicity threat/identity_attack/insult/obscene/sexual_explicit) used to *derive* the rule, and 3 PROSPECTIVE (CAD-IMDB sentiment; CEBaB food/service aspect) predicted *before* their outcome is revealed.

## Setup

### Install dependencies

The router replay only needs **numpy** (for the threshold sweeps / bootstrap CI helpers, copied verbatim from `method.py`) and **matplotlib** (for the figures). Both are pre-installed on Colab, so they are installed locally only — at Colab's exact versions — behind the `google.colab` guard.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# numpy, matplotlib are pre-installed on Colab. Install locally only (at Colab's versions)
# so a local run matches the Colab environment. On Colab this block is skipped.
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'matplotlib==3.10.0')

### Imports

`numpy` + stdlib `json` are exactly what `method.py`'s router section uses; `matplotlib` is added for the demo's figures.

In [ ]:
import json, os
import numpy as np
import matplotlib.pyplot as plt

### Load the precomputed per-concept firing-structure table

`mini_demo_data.json` is a curated subset of the experiment's `method_out.json`. It carries, for all 15 concepts, exactly the fields the router reads (firing-Jaccard median/min/max, recall-hole max, ground-truth regime, and the matched-pool outcome AUCs), plus each concept's per-sub-context firing-Jaccard breakdown and the **original** router results for a match check.

We load it from GitHub with a local fallback (so the notebook works both in Colab and locally).

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-7ee30c-catching-silent-feature-absorption-in-fr/main/round-3/experiment-4/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
meta = data["metadata"]
print("method      :", meta["method_name"])
print("SAE         :", meta["sae_release"], meta["sae_id"])
print("model       :", meta["model"], "| hook:", meta["hook"])
print("gating      : recon_cos_mean=%.4f  L0 median=%.0f" % (
      meta["gating"]["recon_cos_mean"], meta["gating"]["l0_median"]))
print("concepts    :", len(data["concepts"]),
      "(%d established + %d prospective)" % (len(meta["established_concepts"]),
                                            len(meta["prospective_concepts"])))

## Config

All tunable parameters live here. These are the **grid resolutions** for the router's threshold sweeps — the only knobs in the router section of `method.py`. Because the router runs on 15 precomputed numbers it finishes in milliseconds, so we keep the **original full-resolution grids** (the demo fits the runtime budget with huge margin). To run a faster/coarser sweep, just lower the `*_N` values.

`ESTABLISHED` / `PROSPECTIVE` are the concept splits exactly as in `method.py`: τ\* is derived on the established concepts only, then the prospective concepts are predicted out-of-sample.

In [ ]:
# --- router threshold-sweep grids (original method.py values) ---
TAU_LO, TAU_HI, TAU_GRID_N = 0.05, 0.35, 31      # primary firing-Jaccard sweep: np.linspace(0.05, 0.35, 31)
TAU_GRID   = np.linspace(TAU_LO, TAU_HI, TAU_GRID_N)
HOLE_GRID_N      = 41                             # recall-hole-only router resolution (orig: 41)
COMBINED_TAU_J_N = 27                             # 2-signal router, firing-Jaccard axis (orig: 27)
COMBINED_TAU_H_N = 19                             # 2-signal router, recall-hole axis (orig: 19)

# concept splits (as in method.py): derive tau on ESTABLISHED, predict PROSPECTIVE out-of-sample
ESTABLISHED = meta["established_concepts"]
PROSPECTIVE = meta["prospective_concepts"]
print("ESTABLISHED (%d): %s" % (len(ESTABLISHED), ESTABLISHED))
print("PROSPECTIVE (%d): %s" % (len(PROSPECTIVE), PROSPECTIVE))

## Core router functions (copied verbatim from `method.py`)

These are the exact functions from the experiment's router section. The only change is that the hard-coded grid sizes (`41`, `27`, `19`) now read from the config variables above — the logic is untouched.

- `firing_jaccard` — the headline a-priori signal: Jaccard overlap of two latents' firing masks (shown here for reference; the per-concept values were computed on-GPU and are loaded from the data file).
- `balanced_accuracy` — regime-prediction score, averaged over the two regimes.
- `derive_tau` — STEP 3: balanced-accuracy sweep over `TAU_GRID` to pick τ\* (predict `absorption` iff firing-Jaccard < τ) + the regime-separation margins.
- `derive_1d` — single-threshold router on any one signal (used for the recall-hole-only router).
- `derive_combined` — the 2-signal router: `absorption` iff (firing-Jaccard < τ_j) AND (recall_hole_max > τ_h).
- `boot_ci` — percentile bootstrap CI helper.

In [ ]:
def firing_jaccard(fires_a, fires_b):
    a = fires_a.astype(bool); b = fires_b.astype(bool)
    inter = int((a & b).sum()); union = int((a | b).sum())
    return (inter / union) if union > 0 else 0.0


def boot_ci(vals, lo=2.5, hi=97.5):
    vals = np.asarray(vals, dtype=np.float64)
    if len(vals) == 0:
        return [float("nan"), float("nan")]
    return [float(np.percentile(vals, lo)), float(np.percentile(vals, hi))]


def balanced_accuracy(pred, truth):
    pred = np.asarray(pred); truth = np.asarray(truth)
    accs = []
    for cls in ["absorption", "co_firing"]:
        m = truth == cls
        if m.sum() == 0:
            continue
        accs.append((pred[m] == cls).mean())
    return float(np.mean(accs)) if accs else float("nan")


def derive_tau(concepts):
    """Derive tau* by balanced-accuracy sweep over the given concepts (predict absorption iff j<tau)."""
    j = np.array([c["jaccard_median"] for c in concepts])
    truth = np.array([c["ground_truth_regime"] for c in concepts])
    sweep = []
    for tau in TAU_GRID:
        pred = np.where(j < tau, "absorption", "co_firing")
        sweep.append(dict(tau=float(tau), balanced_acc=balanced_accuracy(pred, truth)))
    best = max(sweep, key=lambda r: (r["balanced_acc"], -abs(r["tau"] - 0.15)))
    abs_j = [c["jaccard_median"] for c in concepts if c["ground_truth_regime"] == "absorption"]
    cof_j = [c["jaccard_median"] for c in concepts if c["ground_truth_regime"] == "co_firing"]
    return best["tau"], best["balanced_acc"], sweep, dict(
        max_absorption_j=float(np.max(abs_j)) if abs_j else float("nan"),
        min_cofiring_j=float(np.min(cof_j)) if cof_j else float("nan"),
        mean_absorption_j=float(np.mean(abs_j)) if abs_j else float("nan"),
        mean_cofiring_j=float(np.mean(cof_j)) if cof_j else float("nan"))


def derive_1d(concepts, key, lt):
    """Single-threshold router on `key` (absorption iff value < tau when lt else value > tau)."""
    vals = np.array([c[key] for c in concepts], float)
    truth = np.array([c["ground_truth_regime"] for c in concepts])
    if len(vals) == 0 or not np.isfinite(vals).any():
        return float("nan"), float("nan"), []
    grid = np.linspace(float(np.nanmin(vals)), float(np.nanmax(vals)), HOLE_GRID_N)
    sweep = []
    for tau in grid:
        pred = np.where(vals < tau if lt else vals > tau, "absorption", "co_firing")
        sweep.append(dict(tau=float(tau), balanced_acc=balanced_accuracy(pred, truth)))
    best = max(sweep, key=lambda r: r["balanced_acc"])
    return best["tau"], best["balanced_acc"], sweep


def derive_combined(concepts):
    """2-signal router: absorption iff (firing-Jaccard < tau_j) AND (recall_hole_max > tau_h).
    The honest finding: firing-disjointness ALONE is insufficient -- grouping helps only when the
    parent ALSO has recall holes to fill (taxonomic = low-Jaccard but high-recall parent => no help)."""
    j = np.array([c["jaccard_median"] for c in concepts], float)
    h = np.array([c["recall_hole_max"] for c in concepts], float)
    truth = np.array([c["ground_truth_regime"] for c in concepts])
    if len(j) == 0:
        return dict(tau_j=float("nan"), tau_h=float("nan"), balanced_acc=float("nan"))
    best = None
    for tj in np.linspace(0.05, 0.70, COMBINED_TAU_J_N):
        for th in np.linspace(0.0, 0.9, COMBINED_TAU_H_N):
            pred = np.where((j < tj) & (h > th), "absorption", "co_firing")
            ba = balanced_accuracy(pred, truth)
            if best is None or ba > best["balanced_acc"]:
                best = dict(tau_j=float(tj), tau_h=float(th), balanced_acc=float(ba))
    return best

# quick self-check that firing_jaccard behaves: disjoint -> 0, identical -> 1
_a = np.array([1, 0, 1, 0, 0], bool); _b = np.array([0, 1, 0, 1, 1], bool)
print("firing_jaccard(disjoint) =", firing_jaccard(_a, _b),
      "| firing_jaccard(identical) =", firing_jaccard(_a, _a))

## Assemble the per-concept results

In `method.py` this list is produced by `run_concept(...)` after the GPU forward passes. Here we load it from the cached table — each entry already carries the firing-Jaccard, recall-hole, ground-truth regime, and matched-pool outcome for one concept. We split it into the established set (used to derive the rule) and the prospective set (predicted out-of-sample), exactly as `main()` does.

In [ ]:
results = data["concepts"]                       # 15 per-concept result dicts (precomputed)
est = [r for r in results if r["concept"] in ESTABLISHED]
pro = [r for r in results if r["concept"] in PROSPECTIVE]
print("loaded %d concepts -> %d established, %d prospective" % (len(results), len(est), len(pro)))
for r in est:
    print("  EST  %-26s j_med=%.4f  recall_hole_max=%.3f  truth=%s" % (
          r["concept"], r["jaccard_median"], r["recall_hole_max"], r["ground_truth_regime"]))
for r in pro:
    print("  PRO  %-26s j_med=%.4f  recall_hole_max=%.3f  truth=%s" % (
          r["concept"], r["jaccard_median"], r["recall_hole_max"], r["ground_truth_regime"]))

## STEP 3 — derive the router threshold τ\* (established concepts only)

Sweep τ over `TAU_GRID`, predicting `absorption` iff firing-Jaccard < τ, and pick the τ with the highest balanced accuracy (ties broken toward 0.15). Also report the **regime-separation** margins: the largest firing-Jaccard among absorption concepts vs the smallest among co-firing concepts.

We then verify the re-derived τ\* / balanced accuracy match the values stored from the original GPU run.

In [ ]:
tau_star, bacc, sweep, sep = derive_tau(est)
print("tau*                = %.3f" % tau_star)
print("balanced_acc        = %.4f" % bacc)
print("regime separation   : max_absorption_j=%.4f  min_cofiring_j=%.4f" % (
      sep["max_absorption_j"], sep["min_cofiring_j"]))
print("                      mean_absorption_j=%.4f  mean_cofiring_j=%.4f" % (
      sep["mean_absorption_j"], sep["mean_cofiring_j"]))

ref = meta["router_reference"]
print("\n--- match check vs original GPU run ---")
print("tau*          re-derived=%.3f   original=%.3f   match=%s" % (
      tau_star, ref["tau_star"], abs(tau_star - ref["tau_star"]) < 1e-9))
print("balanced_acc  re-derived=%.4f  original=%.4f  match=%s" % (
      bacc, ref["balanced_acc_at_tau_star"], abs(bacc - ref["balanced_acc_at_tau_star"]) < 1e-9))

## STEP 4 — prospective predict-then-measure

Apply the frozen τ\* to **every** concept to get a predicted regime, then compare against the revealed ground truth. The 3 prospective concepts are the genuine out-of-sample test (their outcome played no role in deriving τ\*).

Honest result: the router separates the *extremes* cleanly but the prospective accuracy is only 1/3 — `sentiment` is a HIT (co_firing predicted & measured) while the two `aspect_*` concepts are MISSES (predicted co_firing but showed small absorption deltas). This is the boundary the paper reports.

In [ ]:
# predicted regime for all concepts under the frozen tau*; hit = matches ground truth
for r in results:
    r["predicted_regime"] = ("absorption" if (np.isfinite(tau_star) and r["jaccard_median"] < tau_star)
                             else "co_firing")
    r["hit"] = bool(r["predicted_regime"] == r["ground_truth_regime"])

prosp_table = [dict(concept=r["concept"], jaccard_median=r["jaccard_median"],
                    predicted_regime=r["predicted_regime"], ground_truth_regime=r["ground_truth_regime"],
                    delta=r["outcome"]["delta"], hit=r["hit"]) for r in pro]
oos_hits = [r["hit"] for r in pro]

print("PROSPECTIVE predict-then-measure")
print("%-16s %9s %12s %12s %8s %5s" % ("concept", "j_median", "predicted", "truth", "delta", "hit"))
for t in prosp_table:
    print("%-16s %9.4f %12s %12s %+8.3f %5s" % (
          t["concept"], t["jaccard_median"], t["predicted_regime"],
          t["ground_truth_regime"], t["delta"], t["hit"]))
oos_acc = float(np.mean(oos_hits)) if oos_hits else float("nan")
print("\nout-of-sample accuracy = %.4f (%d/%d)   [original: %.4f]" % (
      oos_acc, sum(oos_hits), len(oos_hits),
      ref["prospective"]["out_of_sample_accuracy"]))

## STEP 5 — leave-one-concept-out (LOO) cross-validation

For each concept, re-derive τ on **all the others** and predict the held-out concept. This guards against τ\* being tuned to any single concept. (Degenerate folds where the rest are all one regime fall back to the global τ\*, as in `method.py`.)

In [ ]:
loo = []
allc = results
for i, ci in enumerate(allc):
    rest = [c for j, c in enumerate(allc) if j != i]
    if len(set(c["ground_truth_regime"] for c in rest)) < 2:
        tau_i = tau_star                          # degenerate fold -> reuse global tau*
    else:
        tau_i, _, _, _ = derive_tau(rest)
    pred = "absorption" if (np.isfinite(tau_i) and ci["jaccard_median"] < tau_i) else "co_firing"
    loo.append(dict(concept=ci["concept"], tau_fold=float(tau_i), pred=pred,
                    ground_truth=ci["ground_truth_regime"], hit=bool(pred == ci["ground_truth_regime"])))
loo_acc = float(np.mean([x["hit"] for x in loo])) if loo else float("nan")

print("%-26s %8s %12s %12s %5s" % ("concept", "tau_fold", "pred", "truth", "hit"))
for x in loo:
    print("%-26s %8.3f %12s %12s %5s" % (x["concept"], x["tau_fold"], x["pred"], x["ground_truth"], x["hit"]))
print("\nLOO accuracy = %.4f (%d/%d)   [original: %.4f]" % (
      loo_acc, sum(x["hit"] for x in loo), len(loo), ref["loo_accuracy"]))

## Alternative routers — the honest finding

firing-Jaccard *alone* is insufficient. The instructive counterexample is **taxonomic**: it has a *low* firing-Jaccard (narrow country specialists exist) yet a **co_firing** outcome — because the parent already has ~0.95 recall, so there are **no holes to fill** and grouping cannot help.

Two alternatives quantify this:
- **recall-hole-only router** (`derive_1d` on `recall_hole_max`, absorption iff hole > τ_h): reaches balanced accuracy **1.0** on the established concepts.
- **2-signal router** (`derive_combined`): absorption iff (firing-Jaccard < τ_j) AND (recall_hole_max > τ_h) — the refined rule *"grouping helps only when disjoint specialists AND parent recall holes co-occur."*

In [ ]:
tau_h, bacc_hole, sweep_hole = derive_1d(est, "recall_hole_max", lt=False)
combined = derive_combined(est)

print("recall-hole-only router : tau_h=%.4f  balanced_acc=%.4f   [original: %.4f]" % (
      tau_h, bacc_hole, ref["recall_hole_router"]["balanced_acc"]))
print("2-signal combined router: tau_j=%.3f  tau_h=%.3f  balanced_acc=%.4f   [original: %.4f]" % (
      combined["tau_j"], combined["tau_h"], combined["balanced_acc"],
      ref["combined_jaccard_and_hole_router"]["balanced_acc"]))
print("primary firing-Jaccard  :                       balanced_acc=%.4f" % bacc)

## Reproduction check

A sanity check carried in the output: every **spelling** concept has firing-Jaccard < 0.1 (firing-disjoint → absorption regime), reproducing the iter-2 result, while **toxicity** concepts co-fire (Jaccard ≈ 0.69 for threat/insult).

In [ ]:
sp_j = {r["concept"]: r["jaccard_median"] for r in results if r["concept"].startswith("spelling_")}
tox_j = {r["concept"].replace("toxicity_", ""): r["jaccard_median"]
         for r in results if r["concept"].startswith("toxicity_")}
spelling_all_below_0_1 = bool(all(v < 0.10 for v in sp_j.values())) if sp_j else None

print("spelling firing-Jaccard (all should be < 0.1):")
for k, v in sp_j.items():
    print("   %-12s %.4f   %s" % (k, v, "OK" if v < 0.1 else "FAIL"))
print("spelling_all_below_0_1 =", spelling_all_below_0_1,
      "  [original:", meta["reproduction_check"]["spelling_all_below_0_1"], "]")
print("\ntoxicity firing-Jaccard (co-firing regime):")
for k in ["threat", "identity_attack", "insult", "obscene", "sexual_explicit"]:
    if k in tox_j:
        print("   %-16s %.4f" % (k, tox_j[k]))

## Results & visualization

Four views of the router:

1. **Prediction-vs-outcome table** — per concept: firing-Jaccard, predicted vs ground-truth regime, and the matched-pool AUCs (unit / best-single-latent (a) / supervised-attribution (h) / non-SAE probe (d)) with the unit-vs-(a) delta.
2. **Regime separation** — firing-Jaccard per concept (log axis) colored by ground-truth regime, with τ\* marked.
3. **τ sweep** — balanced accuracy vs threshold, peaking at τ\*.
4. **2-signal view** — firing-Jaccard vs recall-hole scatter, exposing the taxonomic counterexample (low Jaccard, low hole → co_firing).

In [ ]:
# ---- 1) prediction-vs-outcome table (mirrors method.py's _print_summary) ----
hdr = "%-26s %7s %11s %11s %6s %6s %6s %6s %7s %5s" % (
    "concept", "j_med", "pred", "truth", "aucU", "aucA", "aucH", "aucD", "dVa", "hit")
print(hdr); print("-" * len(hdr))
for r in results:
    o = r["outcome"]
    print("%-26s %7.3f %11s %11s %6.3f %6.3f %6.3f %6.3f %+7.3f %5s" % (
        r["concept"], r["jaccard_median"], r["predicted_regime"], r["ground_truth_regime"],
        o["auc_unit"], o.get("auc_a", float("nan")), o["auc_h"], o.get("auc_d", float("nan")),
        o["delta"], r["hit"]))
print("\ntau*=%.3f  balanced_acc=%.4f  LOO_acc=%.4f  prospective=%.4f" % (
      tau_star, bacc, loo_acc, oos_acc))

In [ ]:
# ---- 2-4) figures ----
fig, axes = plt.subplots(1, 3, figsize=(18, 5.2))

# (2) regime separation: firing-Jaccard per concept, colored by ground-truth regime
order = sorted(results, key=lambda r: r["jaccard_median"])
names = [r["concept"] for r in order]
jvals = [max(r["jaccard_median"], 1e-3) for r in order]   # floor for log axis
colors = ["#2ca02c" if r["ground_truth_regime"] == "absorption" else "#d62728" for r in order]
edge = ["black" if r["concept"] in PROSPECTIVE else "none" for r in order]
ax = axes[0]
ax.barh(range(len(order)), jvals, color=colors, edgecolor=edge, linewidth=1.6)
ax.set_yticks(range(len(order))); ax.set_yticklabels(names, fontsize=8)
ax.set_xscale("log"); ax.set_xlabel("firing-Jaccard median (log)")
ax.axvline(tau_star, color="navy", ls="--", lw=1.5, label="tau*=%.2f" % tau_star)
ax.set_title("Regime separation\n(green=absorption, red=co_firing; black edge=prospective)", fontsize=10)
ax.legend(loc="lower right", fontsize=9)

# (3) tau sweep: balanced accuracy vs threshold
ax = axes[1]
ts = [s["tau"] for s in sweep]; ba = [s["balanced_acc"] for s in sweep]
ax.plot(ts, ba, "-o", ms=3, color="navy")
ax.axvline(tau_star, color="green", ls="--", lw=1.5, label="tau*=%.2f" % tau_star)
ax.scatter([tau_star], [bacc], color="green", zorder=5, s=60)
ax.set_xlabel("tau (firing-Jaccard threshold)"); ax.set_ylabel("balanced accuracy")
ax.set_title("tau sweep on established concepts", fontsize=10)
ax.set_ylim(0.4, 1.02); ax.legend(fontsize=9); ax.grid(alpha=0.3)

# (4) 2-signal view: firing-Jaccard vs recall-hole, ground-truth colored
ax = axes[2]
for r in results:
    c = "#2ca02c" if r["ground_truth_regime"] == "absorption" else "#d62728"
    mk = "s" if r["concept"] in PROSPECTIVE else "o"
    ax.scatter(max(r["jaccard_median"], 1e-3), r["recall_hole_max"], c=c, marker=mk,
               s=70, edgecolor="black", linewidth=0.5, zorder=3)
ax.axvline(tau_star, color="navy", ls="--", lw=1.2, label="tau*_j=%.2f" % tau_star)
ax.axhline(ref["recall_hole_router"]["tau_h"], color="purple", ls=":", lw=1.2,
           label="tau_h=%.2f" % ref["recall_hole_router"]["tau_h"])
# annotate the taxonomic counterexample
tax = next((r for r in results if r["concept"] == "taxonomic"), None)
if tax is not None:
    ax.annotate("taxonomic\n(low J, low hole -> co_firing)",
                xy=(max(tax["jaccard_median"], 1e-3), tax["recall_hole_max"]),
                xytext=(0.012, 0.45), fontsize=8,
                arrowprops=dict(arrowstyle="->", color="black"))
ax.set_xscale("log"); ax.set_xlabel("firing-Jaccard median (log)")
ax.set_ylabel("recall_hole_max"); ax.set_ylim(-0.05, 1.08)
ax.set_title("2-signal router view\n(absorption needs low J AND high hole)", fontsize=10)
ax.legend(fontsize=8, loc="center right"); ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()

### Per-sub-context firing-Jaccard: a disjoint vs a co-firing concept

The router signal is aggregated from **per-sub-context** detector–parent firing-Jaccards. Here we contrast a spelling concept (every sub-context is firing-*disjoint* from the parent → absorption) against a toxicity concept (sub-contexts *co-fire* with the parent → a single specialist wins). Error bars are the per-sub-context bootstrap CIs carried in the data.

In [ ]:
pcfj = {x["concept"]: x["per_subcontext"] for x in data["per_concept_firing_jaccard"]}

def subcontext_bars(ax, concept, title_regime):
    rows = pcfj.get(concept, [])[:12]          # cap to keep the axis readable
    labs = [str(r["sub_context"]) for r in rows]
    js   = [r["jaccard"] for r in rows]
    los  = [r["jaccard"] - r["jaccard_ci"][0] for r in rows]
    his  = [r["jaccard_ci"][1] - r["jaccard"] for r in rows]
    x = np.arange(len(rows))
    ax.bar(x, js, yerr=[los, his], capsize=2,
           color=("#2ca02c" if title_regime == "absorption" else "#d62728"))
    ax.axhline(tau_star, color="navy", ls="--", lw=1.2, label="tau*=%.2f" % tau_star)
    ax.set_xticks(x); ax.set_xticklabels(labs, rotation=60, ha="right", fontsize=7)
    ax.set_ylabel("firing-Jaccard(detector, parent)")
    ax.set_title("%s (%s regime)" % (concept, title_regime), fontsize=10)
    ax.legend(fontsize=8)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.6))
subcontext_bars(axes[0], "spelling_L", "absorption")
subcontext_bars(axes[1], "toxicity_insult", "co_firing")
plt.tight_layout(); plt.show()

## Summary

Re-running the router on the cached firing-structure reproduces the original GPU run:

- **τ\* = 0.05**, balanced accuracy **0.917** on the 12 established concepts.
- **LOO accuracy 0.733** (leave-one-concept-out).
- **Prospective out-of-sample accuracy 0.333** — `sentiment` HIT, `aspect_food`/`aspect_service` MISS (the honest boundary).
- **recall-hole-only router** reaches balanced accuracy **1.0**, and the **2-signal router** confirms the refined rule: grouping helps only when *firing-disjoint specialists* AND *parent recall holes* co-occur.

The takeaway: a single cheap, **label-free** forward-pass statistic (firing-Jaccard) is a usable a-priori router for *when* cluster-level grouping beats marginal attribution — but only together with the parent's recall-hole structure.